In [ ]:
import os
import polars as pl
import numpy as np
from pathlib import Path
from functools import partial
from collections import defaultdict
from datetime import datetime as dt
import matplotlib.pyplot as plt

from piepy.psychophysics.wheel.wheelTrace import WheelTrace
from piepy.psychophysics.wheel.detection.wheelDetectionExperimentHub import WheelDetectionExperimentHub
from piepy.psychophysics.wheel.detection.wheelDetectionGroupedAggregator import WheelDetectionGroupedAggregator
from piepy.plotters.psychophysics.detection.wheel_profile_plot import *
from piepy.plotters.plotting_utils import (
    set_style,
    make_linear_axis,
    make_label,
    override_plots,
    pval_plotter,
)

In [ ]:
DATA_PATH = Path.cwd().parents[0] / "260410_Ncomm_inhibition_data.parquet"

CONTRAST_COLORS={
        "1.0":{"color":"#262626"},
        "0.5":{"color":"#4d4d4d"},
        "0.25": {"color":"#737373"},
        "0.125":{"color":"#999999"},
        "0.0625":{"color":"#bfbfbf"},
        "0.03125":{"color":"#e6e6e6"},
        "0.0":{"color":"#551177",
               "linestyle":"--"},
        "-1":{"color":"#514577"},
        "100.0":{"color":"#262626"},
        "50.0":{"color":"#4d4d4d"},
        "25.0":{"color":"#737373"},
        "12.5":{"color":"#999999"},
        "6.25":{"color":"#bfbfbf"},
        "3.125":{"color":"#e6e6e6"}
    }

CONTRAST_IDX = {0.0:0,
                0.125:1,
                0.5:2}

In [ ]:
hub = WheelDetectionExperimentHub()
# load from session list
# hub.set_data(all_sessions,
#              load_sessions=True,
#              make_summary=True)
# hub.data.write_parquet("250206_experiment_data.parquet")

# load from saved data
all_data = pl.read_parquet(DATA_PATH)
hub.set_data(all_data,
             load_sessions=True,
             make_summary=True)

In [ ]:
area = "V1"
stim_combination = "0.04cpd_8.0Hz"
df = hub.filter_by_areas(area,
                    strict_performance=True,
                    stim_combination=stim_combination,
                    isCNO=False)
# removing KC139 and 143
# df = df.filter(~pl.col("animalid").is_in(["KC139","KC142"]))
hub.make_summary_data(df)

## AVG of AVGS

In [ ]:
def get_wheel_averages(data:pl.DataFrame,include_misses:bool=False) -> dict:
    """_summary_

    Args:
        data (pl.DataFrame): _description_
    """
    
    analyzer = WheelDetectionGroupedAggregator()
    analyzer.set_data(data=data)
    analyzer.group_data(group_by=["stim_type", 
                                  "stim_side", 
                                  "contrast", 
                                  "opto_pattern", 
                                  "isCatch"],extra_grouped=["t_vstimstart","t_vstimstart_rig","outcome"])
    analyzer.calculate_hit_rates()
    analyzer.calculate_baseline_norm_hit_rates()
    analyzer.calculate_opto_pvalues()
    
    nonearly_data = analyzer.grouped_data.drop_nulls("contrast").filter(pl.col("stim_side") != "ipsi")
    lin_axis_dict = make_linear_axis(nonearly_data, "contrast")
    _lin_axis = [
        float(lin_axis_dict[c]) if c is not None else None
        for c in nonearly_data["contrast"].to_list()
    ]
    nonearly_data = nonearly_data.with_columns(pl.Series("linear_axis", _lin_axis))
    ret = defaultdict(dict)
    for filt_tup in make_subsets(nonearly_data, ["stimkey"], start_enumerate=0):
        filt_df = filt_tup[-1]
        filt_key = filt_tup[1]
        
        if not filt_df.is_empty():
            contrast_label = filt_df["contrast"].to_numpy()
            lin_ax = filt_df["linear_axis"].to_numpy()
            
            ret[filt_key]["avg_wheel_traces"] = {}
            ret[filt_key]["median_hit_reaction_time"] = {}
            for c in contrast_label:
                
                contrast_data = filt_df.filter(pl.col("contrast")==c)
                med_hit_rt = contrast_data[0,"median_hit_reaction_times"]
                
                _hits = contrast_data["outcome"].explode().to_list()
                _times = contrast_data["wheel_t"].explode().to_list()
                _pos = contrast_data["wheel_pos"].explode().to_list()
                _reset = contrast_data["t_vstimstart"].explode().to_list()
                
                # only hit      
                hit_times = [t for t,o in zip(_times,_hits) if o=="hit"]
                hit_pos = [p for p,o in zip(_pos,_hits) if o=="hit"]
                hit_reset = [r for r,o in zip(_reset,_hits) if o=="hit"]
                
                
                if len(hit_times):
                    _trace = get_avg_trace(hit_times,hit_pos,hit_reset)
                    ret[filt_key]["avg_wheel_traces"][c] = _trace
                    ret[filt_key]["linear_axis"] = lin_ax
                    ret[filt_key]["contrast_label"] = contrast_label
                    
                    # also get median hit reaction time
                    ret[filt_key]["median_hit_reaction_time"][c] = med_hit_rt

    return ret

def get_avg_trace(times:list[list],positions:list[list],resets:list[int],**kwargs) -> np.ndarray:
    """
    """
    trace = WheelTrace()
    trace_interp_freq = kwargs.get("interp_freq", 5)
    
    _longest_trace_len = 0
    trials_wheel_list = []
    for t,pos,reset in zip(times,positions,resets):
        
        if len(t) == 0:
            # no wheel movement
            t = [reset, reset + 1000]
            pos = [0, 0]
        t = np.array(t)
        pos = np.array(pos)
            
        _, _, t_interp, tick_interp = trace.reset_and_interpolate(
                t, pos, reset, trace_interp_freq
            )
        # convert the interpolation to rad
        interp_pos = trace.cm_to_rad(trace.ticks_to_cm(tick_interp))

        speed = (
            np.abs(trace.get_filtered_velocity(interp_pos, trace_interp_freq))
            * 1000
        )  # rad/s
        idx_in_time_range = np.where(
            (t_interp >= -200) & (t_interp <= 1500)
        )
        
        if len(idx_in_time_range):
            time_window = t_interp[idx_in_time_range]
            # if plot_speed:
            speed_window = speed[idx_in_time_range]
            y_in_rads = speed_window

            # else:
            #     y_in_rads = interp_pos[idx_in_time_range]
            #     if trial["rig_response_tick"] is not None:
            #         pos_thresh = trace.cm_to_rad(
            #             trace.ticks_to_cm(trial["rig_response_tick"])
            #         )
            #         _rig_response_rad_list.append(pos_thresh)

            trials_wheel_list.append(y_in_rads.tolist())
            # adjusting the longest
            if len(y_in_rads) >= _longest_trace_len:
                _longest_trace_len = len(y_in_rads)
                _longest_time = time_window
    # make a matrix to avreage over the rows, pad with None until reaching longest trace
    all_traces_mat = np.array(
        [xi + [None] * (_longest_trace_len - len(xi)) for xi in trials_wheel_list],
        dtype=float,
    )
    avg = np.nanmean(all_traces_mat, axis=0)
    
    return np.vstack((_longest_time.reshape(1,-1),avg.reshape(1,-1)))
    
def nan_generator(shape:tuple):
    """_summary_

    Args:
        shape (tuple): _description_

    Returns:
        _type_: _description_
    """
    x = np.zeros(shape)
    x[:] = np.nan
    return x

In [ ]:
# get session paths
sessions = df["session_path"].unique(maintain_order=True).to_list()
sessions = [s.split(os.sep)[-1] for s in sessions]
sessions

wheel_dict = defaultdict(partial(nan_generator,(len(sessions),
                                                3, #contrast
                                                2, #wheel_time, wheel_pos
                                                10000)))

rt_dict = defaultdict(partial(nan_generator,(len(sessions),
                                            3, # contrast
                                            )))

animalids = []
unique_session = df["session_id"].unique()
for i,s_id in enumerate(unique_session):
    s_data = df.filter(pl.col("session_id")==s_id)

    aid = s_data["animalid"].unique()[0]

    sesh_dict = get_wheel_averages(s_data)
    for j,k in enumerate(sesh_dict.keys()):
        for idx_c,key_c in enumerate(sesh_dict[k]["avg_wheel_traces"].keys()):
            wheel_dict[k][i,CONTRAST_IDX[key_c],:,:sesh_dict[k]["avg_wheel_traces"][key_c].shape[1]] = sesh_dict[k]["avg_wheel_traces"][key_c]
            rt_dict[k][i,CONTRAST_IDX[key_c]] = sesh_dict[k]["median_hit_reaction_time"][key_c]
    animalids.append(aid) 

In [ ]:
cm = 1/2.54
set_style("print") 
f,axs = plt.subplots(1,len(wheel_dict.keys()),figsize=(24*cm,12*cm))     

# NOTE: The opto contrasts are not always the same as non opto contrasts
# e.g. nonopto=[0,0.125,0.5] and opto=[0,0.125]
# because of this a static, encompassing contrast list is used for coloring the traces
CONTRAST_LABEL = [0.0, 0.125, 0.5]

for key_name,ax in zip(wheel_dict.keys(),axs):
    val = wheel_dict[key_name]
    # wheel
    avg_wheel =  np.nanmean(wheel_dict[key_name],axis=0)
    sem_wheel = stats.sem(wheel_dict[key_name],axis=0,nan_policy="omit")
    
    # reaction times
    median_rt = np.nanmedian(rt_dict[key_name],axis=0)
    sem_rt = stats.sem(rt_dict[key_name],axis=0,nan_policy="omit")
    print(key_name, median_rt,sem_rt)
    for j in range(avg_wheel.shape[0]):
        
        ax.plot(wheel_dict[key_name][0,1,0,:], # common time course
                avg_wheel[j,1,:],   # avg traces of each contrast
                linewidth=2,
                color=CONTRAST_COLORS[str(CONTRAST_LABEL[j])]["color"],
                )
        
        ax.fill_between(
            wheel_dict[key_name][0,1,0,:],
            avg_wheel[j,1,:] - sem_wheel[j,1,:],
            avg_wheel[j,1,:] + sem_wheel[j,1,:],
            color=CONTRAST_COLORS[str(CONTRAST_LABEL[j])]["color"],
            alpha=0.2,
            linewidth=0,
        )
        
        # on the (-) side plot the median reaction times
        
        # session medians as individual ticks
        ax.scatter(rt_dict[key_name][:,j],
                   [-0.5]*rt_dict[key_name].shape[0],
                   marker="|",
                   alpha=0.3,
                   color=CONTRAST_COLORS[str(CONTRAST_LABEL[j])]["color"])
        
        #mean point
        ax.scatter(median_rt[j],-0.5,marker="o",s=10,
                   color=CONTRAST_COLORS[str(CONTRAST_LABEL[j])]["color"])
    
        ax.hlines(-0.5,
                  xmin=median_rt[j]-sem_rt[j],
                  xmax=median_rt[j]+sem_rt[j],
                  color=CONTRAST_COLORS[str(CONTRAST_LABEL[j])]["color"])
    
    ax.axhline(0,color='k',linewidth=1)
    ax.set_title(key_name)
    # ax.set_xticks(sesh_dict[k]["linear_axis"])
    # ax.set_xticklabels(sesh_dict[k]["contrast_label"]) 
    ax.set_ylim([-1, 5])
    ax.set_xlim([-250, 1500])  
    ax.set_yticks([0,1,2,3,4,5])
    ax.set_xticks([0,500,1000,1500])
    ax.set_xlabel("Time from stim onset (ms)")
    ax.set_ylabel("wheel speed (rad/s)")